In [ ]:
  from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls /content/drive/MyDrive

'Colab Notebooks'   dataset


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive

'Colab Notebooks'   dataset


In [ ]:
!ls /content/drive/MyDrive/dataset

train2017  val2017


In [ ]:
!pip install opencv-python tqdm

In [ ]:
import tensorflow as tf
import os
import cv2

IMG_SIZE = 128
BATCH_SIZE = 8
EPOCHS = 100

DATA_PATH = "/content/drive/MyDrive/dataset/train2017"

In [ ]:
def load_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = (img / 127.5) - 1

    gray = tf.image.rgb_to_grayscale(img)

    return gray, img


file_paths = tf.data.Dataset.list_files(DATA_PATH + "/*.jpg", shuffle=True)

dataset = file_paths.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Dataset ready ✅")

Dataset ready ✅


In [ ]:
from tensorflow.keras import layers

def build_generator():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1))

    e1 = layers.Conv2D(64, 4, strides=2, padding='same')(inputs)
    e1 = layers.LeakyReLU()(e1)

    e2 = layers.Conv2D(128, 4, strides=2, padding='same')(e1)
    e2 = layers.BatchNormalization()(e2)
    e2 = layers.LeakyReLU()(e2)

    d1 = layers.Conv2DTranspose(64, 4, strides=2, padding='same')(e2)
    d1 = layers.BatchNormalization()(d1)
    d1 = layers.ReLU()(d1)

    d1 = layers.Concatenate()([d1, e1])

    outputs = layers.Conv2DTranspose(
        3, 4, strides=2,
        padding='same',
        activation='tanh'
    )(d1)

    return tf.keras.Model(inputs, outputs)

generator = build_generator()

In [ ]:
generator.compile(
    optimizer=tf.keras.optimizers.Adam(2e-4, beta_1=0.5),
    loss='mae'
)

In [ ]:
generator.fit(dataset, epochs=EPOCHS)

Epoch 1/100
2046/2046 ━━━━━━━━━━━━━━━━━━━━ 349s 168ms/step - loss: 0.1537
Epoch 2/100
2046/2046 ━━━━━━━━━━━━━━━━━━━━ 101s 49ms/step - loss: 0.1159
Epoch 3/100
2046/2046 ━━━━━━━━━━━━━━━━━━━━ 127s 62ms/step - loss: 0.1122
Epoch 4/100
2046/2046 ━━━━━━━━━━━━━━━━━━━━ 98s 48ms/step - loss: 0.1108
Epoch 5/100
  44/2046 ━━━━━━━━━━━━━━━━━━━━ 1:26 43ms/step - loss: 0.1086

In [ ]:
generator.save("generator_colorization.keras")

In [ ]:
from google.colab import files
files.download("generator_colorization.keras")